# 04 - Unsloth Fine-tuning for Daraja

This notebook fine-tunes Gemma using Unsloth's optimized QLoRA implementation for efficient training.

**Platform:** Kaggle (T4 GPU) or Google Colab (T4/A100)
**GPU Required:** Yes (T4 minimum, 16GB VRAM recommended)
**Estimated Time:** 2-4 hours for full training

## What This Does:
1. Loads Gemma 9B with 4-bit quantization
2. Adds LoRA adapters for efficient fine-tuning
3. Trains on synthetic parallel data
4. Exports to GGUF format for Ollama

## Prerequisites:
- Run `02_synthetic_corpus_generation.ipynb` first to generate training data
- OR have parallel data files ready in the expected format

In [ ]:
# ============================================================
# ENVIRONMENT SETUP
# ============================================================
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

print(f"Running in: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

# Install Unsloth (the order matters!)
if IN_KAGGLE:
    # Kaggle-specific installation
    !pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
    !pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    # Colab / Local installation
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install -q --no-deps xformers trl peft accelerate bitsandbytes

!pip install -q datasets

# Set up directories
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/Daraja'
elif IN_KAGGLE:
    BASE_DIR = '/kaggle/working'
else:
    BASE_DIR = '..'

print(f"Base directory: {BASE_DIR}")

In [ ]:
# ============================================================
# IMPORTS AND CONFIGURATION
# ============================================================
import torch
from pathlib import Path
from datetime import datetime

# Language pair configuration - EDIT THESE FOR YOUR RUN
SOURCE_LANG = "so"           # Source language code
TARGET_LANG = "sw"           # Target language code

# Language names for prompts
LANG_NAMES = {
    "so": "Somali", "sw": "Swahili", "ti": "Tigrinya",
    "ar": "Arabic", "prs": "Dari", "tr": "Turkish"
}
SOURCE_NAME = LANG_NAMES.get(SOURCE_LANG, SOURCE_LANG)
TARGET_NAME = LANG_NAMES.get(TARGET_LANG, TARGET_LANG)

# Paths
DATA_DIR = Path(BASE_DIR) / 'pipeline' / 'data' / 'synthetic' / f'{SOURCE_LANG}-{TARGET_LANG}'
OUTPUT_DIR = Path(BASE_DIR) / 'outputs' / f'{SOURCE_LANG}-{TARGET_LANG}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training hyperparameters
BASE_MODEL = "unsloth/gemma-2-9b-it-bnb-4bit"  # 4-bit quantized Gemma
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 16
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
WARMUP_STEPS = 100

print(f"Configuration:")
print(f"  Language pair: {SOURCE_NAME} -> {TARGET_NAME}")
print(f"  Base model: {BASE_MODEL}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# ============================================================
# LOAD MODEL WITH LORA ADAPTERS
# ============================================================
from unsloth import FastLanguageModel

print(f"Loading model: {BASE_MODEL}")
print("This may take a few minutes...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect (bfloat16 on Ampere+, float16 otherwise)
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory optimization
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print(f"\nModel loaded with LoRA adapters!")
print(f"  Trainable parameters: {model.print_trainable_parameters()}")

In [ ]:
# ============================================================
# PREPARE TRAINING DATA
# ============================================================
from datasets import Dataset

# Check for data files
source_file = DATA_DIR / f'synthetic.{SOURCE_LANG}'
target_file = DATA_DIR / f'synthetic.{TARGET_LANG}'

if not source_file.exists():
    print(f"ERROR: Training data not found at {DATA_DIR}")
    print("Please run 02_synthetic_corpus_generation.ipynb first!")
    raise FileNotFoundError(f"Missing {source_file}")

# Load parallel data
with open(source_file, 'r', encoding='utf-8') as sf:
    sources = sf.read().strip().split('\n')
with open(target_file, 'r', encoding='utf-8') as tf:
    targets = tf.read().strip().split('\n')

# Ensure lengths match
min_len = min(len(sources), len(targets))
sources = sources[:min_len]
targets = targets[:min_len]

print(f"Loaded {len(sources):,} training examples")

# Format training examples using Gemma chat template
def format_example(source, target, domain="humanitarian"):
    """Format a single training example."""
    domain_tag = f"[{domain.upper()}] " if domain else ""
    
    return f"""<start_of_turn>user
Translate the following {SOURCE_NAME} text to {TARGET_NAME}:
{domain_tag}{source}<end_of_turn>
<start_of_turn>model
{target}<end_of_turn>"""

# Create formatted dataset
formatted_examples = [
    format_example(src, tgt)
    for src, tgt in zip(sources, targets)
]

dataset = Dataset.from_dict({"text": formatted_examples})

# Split for validation
split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f"\nDataset prepared:")
print(f"  Training samples: {len(train_dataset):,}")
print(f"  Validation samples: {len(eval_dataset):,}")
print(f"\nSample training example:")
print("-" * 50)
print(train_dataset[0]['text'][:500])
print("-" * 50)

In [ ]:
# ============================================================
# CONFIGURE AND RUN TRAINING
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments
import time

# Training arguments
training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    output_dir=str(OUTPUT_DIR),
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    report_to="none",  # Disable wandb etc.
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

print("Starting training...")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  Total steps: ~{len(train_dataset) * NUM_EPOCHS // (BATCH_SIZE * GRADIENT_ACCUMULATION)}")

start_time = time.time()

# Train!
trainer_stats = trainer.train()

training_time = time.time() - start_time
print(f"\nTraining complete!")
print(f"  Time: {training_time / 60:.1f} minutes")
print(f"  Final loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# ============================================================
# TEST THE TRAINED MODEL
# ============================================================
# Quick test before saving
FastLanguageModel.for_inference(model)

test_sentences = [
    "Hello, how are you?",
    "I need medical help.",
    "Where is the registration office?",
]

print("Testing trained model:")
print("-" * 50)

for test in test_sentences:
    prompt = f"""<start_of_turn>user
Translate the following English text to {TARGET_NAME}:
{test}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.3,
            do_sample=True,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the model's response
    if "<start_of_turn>model" in response:
        response = response.split("<start_of_turn>model")[-1].strip()
    
    print(f"English: {test}")
    print(f"{TARGET_NAME}: {response}")
    print("-" * 50)

In [ ]:
# ============================================================
# SAVE MODEL AND EXPORT TO GGUF
# ============================================================
FINAL_DIR = OUTPUT_DIR / 'final'
FINAL_DIR.mkdir(exist_ok=True)

print("Saving model...")

# 1. Save LoRA adapters (small, for future merging)
lora_dir = FINAL_DIR / 'lora_adapters'
model.save_pretrained(str(lora_dir))
tokenizer.save_pretrained(str(lora_dir))
print(f"  LoRA adapters saved to: {lora_dir}")

# 2. Save merged model (full model with LoRA merged)
merged_dir = FINAL_DIR / 'merged'
model.save_pretrained_merged(
    str(merged_dir),
    tokenizer,
    save_method="merged_16bit"
)
print(f"  Merged model saved to: {merged_dir}")

# 3. Export to GGUF for Ollama (q4_k_m quantization)
gguf_dir = FINAL_DIR / 'gguf'
print(f"  Exporting to GGUF (q4_k_m)... this may take a few minutes")
model.save_pretrained_gguf(
    str(gguf_dir),
    tokenizer,
    quantization_method="q4_k_m"
)
print(f"  GGUF exported to: {gguf_dir}")

# List exported files
print(f"\nExported files:")
for f in gguf_dir.glob("*.gguf"):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}: {size_mb:.1f} MB")

In [ ]:
# ============================================================
# GENERATE OLLAMA MODELFILE
# ============================================================
# Find the GGUF file
gguf_files = list(gguf_dir.glob("*.gguf"))
if not gguf_files:
    print("Warning: No GGUF file found!")
else:
    gguf_path = gguf_files[0]
    modelfile_path = OUTPUT_DIR / f'daraja-{SOURCE_LANG}-{TARGET_LANG}.Modelfile'
    
    modelfile_content = f"""# Daraja Translation Model: {SOURCE_NAME} -> {TARGET_NAME}
# Fine-tuned Gemma model for humanitarian translation
#
# Usage:
#   ollama create daraja-{SOURCE_LANG}-{TARGET_LANG} -f {modelfile_path.name}
#   ollama run daraja-{SOURCE_LANG}-{TARGET_LANG} "Translate: <text>"

FROM {gguf_path}

TEMPLATE \"\"\"<start_of_turn>user
{{{{ .Prompt }}}}<end_of_turn>
<start_of_turn>model
\"\"\"

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER repeat_penalty 1.1
PARAMETER stop "<end_of_turn>"
PARAMETER num_ctx 2048

SYSTEM \"\"\"You are Daraja, a professional translation assistant specializing in {SOURCE_NAME} to {TARGET_NAME} translation for humanitarian contexts.

Your role:
- Translate {SOURCE_NAME} text to {TARGET_NAME} accurately and faithfully
- Preserve the exact meaning and tone of the original
- Handle medical, legal, and administrative terminology with precision
- Never add explanations or commentary - only provide the translation

When given a translation request, output only the {TARGET_NAME} translation.\"\"\"
"""
    
    with open(modelfile_path, 'w') as f:
        f.write(modelfile_content)
    
    print(f"Modelfile generated: {modelfile_path}")
    print(f"\nTo create Ollama model:")
    print(f"  1. Copy {gguf_path.name} to your local machine")
    print(f"  2. Copy {modelfile_path.name} to the same directory")
    print(f"  3. Run: ollama create daraja-{SOURCE_LANG}-{TARGET_LANG} -f {modelfile_path.name}")

In [ ]:
# ============================================================
# SUMMARY
# ============================================================
import json

print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"\nLanguage pair: {SOURCE_NAME} ({SOURCE_LANG}) -> {TARGET_NAME} ({TARGET_LANG})")
print(f"Base model: {BASE_MODEL}")
print(f"Training samples: {len(train_dataset):,}")
print(f"Training time: {training_time / 60:.1f} minutes")
print(f"Final loss: {trainer_stats.training_loss:.4f}")

print(f"\nOutput files:")
print(f"  LoRA adapters: {lora_dir}")
print(f"  Merged model: {merged_dir}")
print(f"  GGUF model: {gguf_dir}")
print(f"  Modelfile: {modelfile_path}")

# Save training summary
summary = {
    "completed_at": datetime.now().isoformat(),
    "language_pair": f"{SOURCE_LANG}-{TARGET_LANG}",
    "source_lang": SOURCE_NAME,
    "target_lang": TARGET_NAME,
    "base_model": BASE_MODEL,
    "training_samples": len(train_dataset),
    "validation_samples": len(eval_dataset),
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "final_loss": trainer_stats.training_loss,
    "training_time_minutes": training_time / 60,
}

summary_path = OUTPUT_DIR / 'training_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"  Summary: {summary_path}")

print(f"\n" + "=" * 60)
print("NEXT STEPS")
print("=" * 60)
print("1. Download the GGUF file and Modelfile to your local machine")
print("2. Install Ollama: https://ollama.ai/download")
print(f"3. Create the model: ollama create daraja-{SOURCE_LANG}-{TARGET_LANG} -f Modelfile")
print(f"4. Test: ollama run daraja-{SOURCE_LANG}-{TARGET_LANG} \"Translate: Hello\"")
print("5. (Optional) Run 05_evaluation.ipynb to measure BLEU/chrF++ scores")
print("=" * 60)

# Cleanup
del model, tokenizer, trainer
import gc
gc.collect()
torch.cuda.empty_cache()
print("\nGPU memory released.")